# LeetCode 1876 — Substrings of Size Three with Distinct Characters bằng Python

Link đề: https://leetcode.com/problems/substrings-of-size-three-with-distinct-characters/

Độ khó: **Easy** · Chủ đề: Hash Table, String, Sliding Window, Counting

Skeleton chữ ký LeetCode:

```python
class Solution:
    def countGoodSubstrings(self, s: str) -> int:

```

**Mục tiêu học**

1. Hiểu **chuỗi con** (substring) liên tiếp và cách liệt kê mọi chuỗi con độ dài 3 bằng slicing `s[i:i+3]`.
2. Ôn `set` để kiểm "không trùng", `zip` để ghép ba ký tự kề nhau.
3. Học 4 cách: slicing + `set` → so sánh trực tiếp 3 ký tự → `zip(s, s[1:], s[2:])` → **cửa sổ trượt** (sliding window) có bộ đếm, tổng quát cho độ dài `k`.
4. Đếm đúng số chỉ số bắt đầu: `len(s) - 2` (và trường hợp chuỗi ngắn hơn 3).
5. Trace, chứng minh, Big-O và 6 bài tập về cửa sổ cố định.

## 1. Bài toán yêu cầu gì?

Một chuỗi là **good** nếu không có ký tự nào lặp lại.

Cho `s`, đếm số **chuỗi con độ dài 3** là good. Nếu cùng một chuỗi con xuất hiện nhiều lần (ở vị trí khác nhau), **mỗi lần đều được đếm**.

**Chuỗi con** = dãy ký tự **liên tiếp** trong `s`.

| s | Các chuỗi con độ dài 3 | Good | Kết quả |
|---|---|---|---:|
| `"xyzzaz"` | xyz, yzz, zza, zaz | xyz | 1 |
| `"aababcabc"` | aab, aba, bab, abc, bca, cab, abc | abc, bca, cab, abc | 4 |

**Constraints:** `1 <= len(s) <= 100`, chỉ chữ thường. Chú ý `len(s)` có thể là 1 hoặc 2 ⇒ không có chuỗi con độ dài 3 ⇒ đáp án 0.

In [1]:
examples = [("xyzzaz", 1), ("aababcabc", 4)]
for s, expected in examples:
    print(repr(s), "->", expected)

'xyzzaz' -> 1
'aababcabc' -> 4


## 2. Nền tảng Python

### 2.1. Slicing `s[i:i+3]` và số chỉ số bắt đầu

`s[i:i+3]` lấy 3 ký tự bắt đầu ở `i`. Chỉ số bắt đầu hợp lệ: `i = 0, 1, ..., len(s) - 3` ⇒ dùng `range(len(s) - 2)`.

In [2]:
s = "xyzzaz"
print("len =", len(s), "-> số chuỗi con độ dài 3 =", len(s) - 2)
for i in range(len(s) - 2):
    print(i, s[i:i + 3])

len = 6 -> số chuỗi con độ dài 3 = 4
0 xyz
1 yzz
2 zza
3 zaz


Nếu `len(s) < 3`, `range(len(s) - 2)` là `range(0)` hoặc `range(-1)` — **rỗng**, nên vòng lặp không chạy và đáp án tự động là 0.

In [3]:
for s in ["a", "ab", "abc"]:
    print(repr(s), list(range(len(s) - 2)))
print("slicing vượt biên không lỗi:", repr("ab"[1:4]))

'a' []
'ab' []
'abc' [0]
slicing vượt biên không lỗi: 'b'


Slicing **không** báo lỗi khi vượt biên — `"ab"[1:4]` chỉ trả `"b"`. Nếu duyệt sai `range(len(s))`, ta sẽ nhận các chuỗi con **ngắn hơn 3** mà không hề có lỗi để cảnh báo.

### 2.2. `set` để kiểm "không trùng"

`len(set(t)) == len(t)` ⇔ mọi ký tự của `t` khác nhau. Với `t` dài 3: `len(set(t)) == 3`.

In [4]:
for t in ["xyz", "yzz", "aba", "abc"]:
    print(t, "-> set =", sorted(set(t)), "| good?", len(set(t)) == 3)

xyz -> set = ['x', 'y', 'z'] | good? True
yzz -> set = ['y', 'z'] | good? False
aba -> set = ['a', 'b'] | good? False
abc -> set = ['a', 'b', 'c'] | good? True


### 2.3. Đếm bằng `sum(điều_kiện for ...)`

`True` được tính là 1, `False` là 0 ⇒ `sum(cond for ...)` đếm số lần điều kiện đúng.

In [5]:
print(sum(x > 2 for x in [1, 3, 5, 2]))      # 2 phần tử > 2
print(True + True + False)

2
2


### 2.4. `zip` ba chuỗi lệch nhau

`zip(s, s[1:], s[2:])` ghép bộ ba `(s[i], s[i+1], s[i+2])`; `zip` dừng ở chuỗi **ngắn nhất** (`s[2:]`) nên tự có đúng `len(s) - 2` bộ.

In [6]:
s = "xyzzaz"
print(list(zip(s, s[1:], s[2:])))

[('x', 'y', 'z'), ('y', 'z', 'z'), ('z', 'z', 'a'), ('z', 'a', 'z')]


## 3. Cách 1 — slicing + `set` (trực giác)

Liệt kê từng chuỗi con độ dài 3, kiểm bằng `set`.

In [7]:
def count_good_set(s: str) -> int:
    count = 0
    for i in range(len(s) - 2):
        if len(set(s[i:i + 3])) == 3:
            count += 1
    return count

for s, expected in examples:
    print(s, "->", count_good_set(s), "| expected", expected)
    assert count_good_set(s) == expected

xyzzaz -> 1 | expected 1
aababcabc -> 4 | expected 4


Cách này đã `O(n)` (vì mỗi chuỗi con chỉ dài 3 — hằng số). Không có "nút thắt" lớn ở bài gốc. Nhưng nếu đề đổi độ dài 3 thành `k` bất kỳ, mỗi chuỗi con tốn `O(k)` để cắt và tạo set ⇒ `O(n·k)`. Ta sẽ học **cửa sổ trượt** để đưa về `O(n)` cho mọi `k`.

## 4. Cách 2 — so sánh trực tiếp ba ký tự

Với 3 ký tự `a, b, c`, "đôi một khác nhau" ⇔ `a != b and b != c and a != c`. Không tạo chuỗi con, không tạo set — nhanh và rõ ràng.

In [8]:
def count_good_compare(s: str) -> int:
    count = 0
    for i in range(len(s) - 2):
        a, b, c = s[i], s[i + 1], s[i + 2]
        if a != b and b != c and a != c:
            count += 1
    return count

for s, expected in examples:
    assert count_good_compare(s) == expected
print(count_good_compare("aababcabc"))

4


Đừng quên cặp `a != c`! Chuỗi `"aba"` có `a != b` và `b != c` nhưng `a == c`.

In [9]:
a, b, c = "aba"
print("a!=b:", a != b, "| b!=c:", b != c, "| a!=c:", a != c, "-> good?", a != b and b != c and a != c)

a!=b: True | b!=c: True | a!=c: False -> good? False


`a, b, c = "aba"` **mở gói** chuỗi 3 ký tự vào 3 biến.

## 5. TRACE từng bước

Trace `s = "aababcabc"`:

| i | s[i:i+3] | a≠b | b≠c | a≠c | good? | count |
|---:|---|:---:|:---:|:---:|:---:|---:|
| 0 | aab | ✗ | ✓ | ✓ | ✗ | 0 |
| 1 | aba | ✓ | ✓ | ✗ | ✗ | 0 |
| 2 | bab | ✓ | ✓ | ✗ | ✗ | 0 |
| 3 | abc | ✓ | ✓ | ✓ | ✓ | 1 |
| 4 | bca | ✓ | ✓ | ✓ | ✓ | 2 |
| 5 | cab | ✓ | ✓ | ✓ | ✓ | 3 |
| 6 | abc | ✓ | ✓ | ✓ | ✓ | 4 |

Chuỗi con `"abc"` xuất hiện 2 lần (i = 3 và i = 6) và được đếm **cả hai**.

In [10]:
def trace_count_good(s: str) -> int:
    count = 0
    print(" i | sub | a!=b | b!=c | a!=c | good | count")
    for i in range(len(s) - 2):
        a, b, c = s[i], s[i + 1], s[i + 2]
        good = a != b and b != c and a != c
        count += good
        mark = lambda v: "✓" if v else "✗"
        print(f"{i:>2} | {s[i:i+3]} |  {mark(a != b)}   |  {mark(b != c)}   |  {mark(a != c)}   |  {mark(good)}   | {count}")
    print("=>", count)
    return count

assert trace_count_good("aababcabc") == 4

 i | sub | a!=b | b!=c | a!=c | good | count
 0 | aab |  ✗   |  ✓   |  ✓   |  ✗   | 0
 1 | aba |  ✓   |  ✓   |  ✗   |  ✗   | 0
 2 | bab |  ✓   |  ✓   |  ✗   |  ✗   | 0
 3 | abc |  ✓   |  ✓   |  ✓   |  ✓   | 1
 4 | bca |  ✓   |  ✓   |  ✓   |  ✓   | 2
 5 | cab |  ✓   |  ✓   |  ✓   |  ✓   | 3
 6 | abc |  ✓   |  ✓   |  ✓   |  ✓   | 4
=> 4


`count += good` cộng `True` (1) hoặc `False` (0).

Trace thứ hai: chuỗi quá ngắn — vòng lặp không chạy lần nào.

In [11]:
assert trace_count_good("ab") == 0
assert trace_count_good("xyzzaz") == 1

 i | sub | a!=b | b!=c | a!=c | good | count
=> 0
 i | sub | a!=b | b!=c | a!=c | good | count
 0 | xyz |  ✓   |  ✓   |  ✓   |  ✓   | 1
 1 | yzz |  ✓   |  ✗   |  ✓   |  ✗   | 1
 2 | zza |  ✗   |  ✓   |  ✓   |  ✗   | 1
 3 | zaz |  ✓   |  ✓   |  ✗   |  ✗   | 1
=> 1


## 6. Cách 3 — `zip` một dòng

In [12]:
def count_good_zip(s: str) -> int:
    return sum(a != b and b != c and a != c for a, b, c in zip(s, s[1:], s[2:]))

for s, expected in examples:
    assert count_good_zip(s) == expected
print(count_good_zip("xyzzaz"), count_good_zip("aababcabc"), count_good_zip("a"))

1 4 0


Một biến thể: `len({a, b, c}) == 3` — tạo set từ 3 ký tự bằng cú pháp `{...}`.

In [13]:
print(sum(len({a, b, c}) == 3 for a, b, c in zip("aababcabc", "aababcabc"[1:], "aababcabc"[2:])))

4


### 6.1. Insight — cửa sổ trượt (sliding window) cho độ dài `k`

Khi độ dài là `k` (không chỉ 3), thay vì xây lại set cho mỗi chuỗi con, ta giữ một **cửa sổ** dài `k` và **trượt** nó sang phải từng bước:

- Thêm ký tự mới vào bên phải: `cnt[s[right]] += 1`.
- Bỏ ký tự cũ ở bên trái: `cnt[s[left]] -= 1` (nếu về 0 thì xóa khỏi dict).
- Cửa sổ good ⇔ **số ký tự khác nhau** trong cửa sổ = `k` ⇔ `len(cnt) == k`.

Mỗi bước chỉ tốn `O(1)` ⇒ tổng `O(n)` bất kể `k`.

`collections.Counter` là dict đếm: `cnt[x]` mặc định 0 nếu chưa có.

In [14]:
from collections import Counter
cnt = Counter()
for ch in "aab":
    cnt[ch] += 1
print(cnt, "| số ký tự khác nhau =", len(cnt))
cnt["a"] -= 1
cnt["a"] -= 1
print("sau khi trừ 'a' về 0:", cnt, "| len =", len(cnt), "(vẫn còn key 'a' với giá trị 0!)")
del cnt["a"]
print("sau khi del:", cnt, "| len =", len(cnt))

Counter({'a': 2, 'b': 1}) | số ký tự khác nhau = 2
sau khi trừ 'a' về 0: Counter({'b': 1, 'a': 0}) | len = 2 (vẫn còn key 'a' với giá trị 0!)
sau khi del: Counter({'b': 1}) | len = 1


Bài học: khi đếm về 0 phải **xóa key**, nếu không `len(cnt)` sẽ đếm cả ký tự đã rời cửa sổ.

## 7. Lời giải tổng quát — cửa sổ trượt, giảng từng đoạn

**Đoạn 1:** `cnt = Counter()`, `count = 0`.

**Đoạn 2:** duyệt `right` từ 0: thêm `s[right]`.

**Đoạn 3:** nếu cửa sổ dài hơn `k` (`right >= k`), bỏ `s[right - k]` (ký tự vừa rơi khỏi bên trái).

**Đoạn 4:** nếu cửa sổ đủ dài `k` (`right >= k - 1`) và `len(cnt) == k` thì `count += 1`.

In [15]:
def count_good_window(s: str, k: int = 3) -> int:
    cnt = Counter()
    count = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        if right >= k:                      # bỏ ký tự rơi khỏi cửa sổ
            old = s[right - k]
            cnt[old] -= 1
            if cnt[old] == 0:
                del cnt[old]
        if right >= k - 1 and len(cnt) == k:
            count += 1
    return count

for s, expected in examples:
    assert count_good_window(s) == expected
print(count_good_window("aababcabc"), "| k=2:", count_good_window("aababcabc", 2), "| k=4:", count_good_window("abcdabc", 4))

4 | k=2: 7 | k=4: 4


In [16]:
def trace_window(s, k=3):
    cnt = Counter()
    count = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        removed = ""
        if right >= k:
            old = s[right - k]
            cnt[old] -= 1
            if cnt[old] == 0:
                del cnt[old]
            removed = f"bỏ {old!r}"
        window = s[max(0, right - k + 1): right + 1]
        good = right >= k - 1 and len(cnt) == k
        count += good
        print(f"right={right} thêm {ch!r} {removed:<7} cửa sổ={window!r:<6} cnt={dict(cnt)} good={good} count={count}")
    return count

assert trace_window("xyzzaz") == 1

right=0 thêm 'x'         cửa sổ='x'    cnt={'x': 1} good=False count=0
right=1 thêm 'y'         cửa sổ='xy'   cnt={'x': 1, 'y': 1} good=False count=0
right=2 thêm 'z'         cửa sổ='xyz'  cnt={'x': 1, 'y': 1, 'z': 1} good=True count=1
right=3 thêm 'z' bỏ 'x'  cửa sổ='yzz'  cnt={'y': 1, 'z': 2} good=False count=1
right=4 thêm 'a' bỏ 'y'  cửa sổ='zza'  cnt={'z': 2, 'a': 1} good=False count=1
right=5 thêm 'z' bỏ 'z'  cửa sổ='zaz'  cnt={'z': 2, 'a': 1} good=False count=1


Quan sát cột `cnt`: khi cửa sổ là `"yzz"`, `cnt = {'y':1, 'z':2}` — chỉ 2 ký tự khác nhau ⇒ không good. Khi bỏ `'y'` (về 0) ta xóa key để `len(cnt)` đúng.

## 8. Code nộp LeetCode

### Cách A — so sánh trực tiếp (khuyên dùng cho k = 3)

In [17]:
class Solution:
    def countGoodSubstrings(self, s: str) -> int:
        count = 0
        for i in range(len(s) - 2):
            a, b, c = s[i], s[i + 1], s[i + 2]
            if a != b and b != c and a != c:
                count += 1
        return count

print(Solution().countGoodSubstrings("xyzzaz"), Solution().countGoodSubstrings("aababcabc"))

1 4


### Cách B — slicing + set

In [18]:
class SolutionSet:
    def countGoodSubstrings(self, s: str) -> int:
        return sum(len(set(s[i:i + 3])) == 3 for i in range(len(s) - 2))

print(SolutionSet().countGoodSubstrings("aababcabc"))

4


### Cách C — cửa sổ trượt (tổng quát)

In [19]:
class SolutionWindow:
    def countGoodSubstrings(self, s: str) -> int:
        k = 3
        cnt = {}
        count = 0
        for right, ch in enumerate(s):
            cnt[ch] = cnt.get(ch, 0) + 1
            if right >= k:
                old = s[right - k]
                cnt[old] -= 1
                if cnt[old] == 0:
                    del cnt[old]
            if right >= k - 1 and len(cnt) == k:
                count += 1
        return count

print(SolutionWindow().countGoodSubstrings("aababcabc"))

4


## 9. Vì sao đúng?

**Cách so sánh:** mỗi chỉ số bắt đầu `i ∈ [0, n-3]` ứng với **đúng một** chuỗi con độ dài 3 (song ánh), nên duyệt `i` là đếm mỗi lần xuất hiện đúng một lần. Ba ký tự đôi một khác nhau ⇔ ba bất đẳng thức `a≠b, b≠c, a≠c` cùng đúng (có đúng 3 cặp từ 3 phần tử).

**Cửa sổ trượt — invariant:** sau khi xử lý `right`, `cnt` chứa **chính xác** số lần xuất hiện của mỗi ký tự trong `s[max(0, right-k+1) .. right]`, và **không có key giá trị 0**. Do đó `len(cnt)` = số ký tự khác nhau trong cửa sổ. Cửa sổ dài `k` là good ⇔ có `k` ký tự khác nhau ⇔ `len(cnt) == k`.

In [20]:
# Kiểm chứng invariant của cửa sổ trên dữ liệu ngẫu nhiên
import random
rng = random.Random(1876)
for _ in range(300):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 30)))
    k = rng.randint(1, 5)
    cnt = Counter()
    for right, ch in enumerate(s):
        cnt[ch] += 1
        if right >= k:
            old = s[right - k]
            cnt[old] -= 1
            if cnt[old] == 0:
                del cnt[old]
        assert cnt == Counter(s[max(0, right - k + 1): right + 1])
        assert 0 not in cnt.values()
print("invariant cửa sổ đúng trên 300 chuỗi")

invariant cửa sổ đúng trên 300 chuỗi


## 10. Độ phức tạp

Gọi `n = len(s)`, `k` = độ dài chuỗi con (bài gốc `k = 3`).

| Cách | Time | Space |
|---|---|---|
| Slicing + set | `O(n·k)` (= `O(n)` khi k = 3) | `O(k)` |
| So sánh trực tiếp | `O(n)` | `O(1)` |
| `zip` | `O(n)` | `O(n)` (các bản sao `s[1:]`, `s[2:]`) |
| Cửa sổ trượt | `O(n)` mọi `k` | `O(min(k, Σ))` (Σ = 26) |

Với `n <= 100` mọi cách đều tức thì; cửa sổ trượt là kỹ thuật **mở rộng được**.

## 11. Bộ test

Nhóm: mẫu đề; chuỗi dài 1, 2; dài đúng 3 (good / không good); toàn một ký tự; lặp lại chuỗi good (đếm mọi lần); ngẫu nhiên đối chiếu 7 cài đặt.

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS ({actual})")

sol = Solution()
check("mẫu 1", sol.countGoodSubstrings("xyzzaz"), 1)
check("mẫu 2", sol.countGoodSubstrings("aababcabc"), 4)
check("dài 1", sol.countGoodSubstrings("a"), 0)
check("dài 2", sol.countGoodSubstrings("ab"), 0)
check("dài 3 good", sol.countGoodSubstrings("abc"), 1)
check("dài 3 'aba'", sol.countGoodSubstrings("aba"), 0)
check("toàn 'a'", sol.countGoodSubstrings("a" * 100), 0)
check("'abc' x 3", sol.countGoodSubstrings("abcabcabc"), 7)

mẫu 1                  PASS (1)
mẫu 2                  PASS (4)
dài 1                  PASS (0)
dài 2                  PASS (0)
dài 3 good             PASS (1)
dài 3 'aba'            PASS (0)
toàn 'a'               PASS (0)
'abc' x 3              PASS (7)


In [22]:
impls = [count_good_set, count_good_compare, count_good_zip, count_good_window,
         Solution().countGoodSubstrings, SolutionSet().countGoodSubstrings, SolutionWindow().countGoodSubstrings]
for _ in range(1500):
    s = "".join(rng.choice("abcde"[: rng.randint(1, 5)]) for _ in range(rng.randint(1, 60)))
    res = {f(s) for f in impls}
    assert len(res) == 1, (s, res)
print("7 cài đặt đồng ý trên 1500 chuỗi ngẫu nhiên PASS")

7 cài đặt đồng ý trên 1500 chuỗi ngẫu nhiên PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: quên so sánh `a != c`

In [23]:
def wrong_two_checks(s):
    return sum(s[i] != s[i + 1] and s[i + 1] != s[i + 2] for i in range(len(s) - 2))

print("sai :", wrong_two_checks("aba"), "| đúng:", count_good_compare("aba"))

sai : 1 | đúng: 0


Sửa: đủ 3 cặp, hoặc dùng `len({a, b, c}) == 3`.

### Lỗi 2: duyệt `range(len(s))` ⇒ đếm cả chuỗi con ngắn

In [24]:
def wrong_range(s):
    count = 0
    for i in range(len(s)):
        t = s[i:i + 3]                 # ở cuối chuỗi t ngắn hơn 3
        if len(set(t)) == len(t):      # "không trùng" nhưng không đủ 3 ký tự
            count += 1
    return count

print("sai :", wrong_range("xyzzaz"), "| đúng:", count_good_set("xyzzaz"))

sai : 3 | đúng: 1


Slicing không báo lỗi khi vượt biên nên bug này "im lặng". Sửa: `range(len(s) - 2)` và kiểm `== 3`.

### Lỗi 3: đếm chuỗi con **khác nhau** thay vì mọi lần xuất hiện

In [25]:
def wrong_unique(s):
    seen = set()
    for i in range(len(s) - 2):
        t = s[i:i + 3]
        if len(set(t)) == 3:
            seen.add(t)
    return len(seen)

print("sai :", wrong_unique("aababcabc"), "| đúng:", count_good_set("aababcabc"))

sai : 3 | đúng: 4


Đề: "every occurrence should be counted". Sửa: đếm bằng biến `count`, không gom vào set.

### Lỗi 4: cửa sổ trượt không xóa key về 0

In [26]:
def wrong_window_no_delete(s, k=3):
    cnt = Counter()
    count = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        if right >= k:
            cnt[s[right - k]] -= 1        # không xóa khi về 0
        if right >= k - 1 and len(cnt) == k:
            count += 1
    return count

print("sai :", wrong_window_no_delete("abcd"), "| đúng:", count_good_window("abcd"))

sai : 1 | đúng: 2


Ở `"bcd"`, `cnt = {'a':0, 'b':1, 'c':1, 'd':1}` ⇒ `len = 4 ≠ 3` ⇒ bị bỏ sót. Sửa: `del cnt[old]` khi về 0 (hoặc đếm số key có giá trị > 0).

## 13. Mẹo nhận dạng pattern

- "Chuỗi con / mảng con **độ dài cố định k**" ⇒ **cửa sổ trượt cố định**: thêm phải, bỏ trái, cập nhật trạng thái `O(1)`.
- "Không có ký tự trùng" ⇒ `set` hoặc dict đếm, so `len(...)` với độ dài.
- `k` rất nhỏ (2, 3) ⇒ so sánh trực tiếp là đơn giản nhất.
- Số chuỗi con độ dài `k` trong chuỗi dài `n` là `max(0, n - k + 1)`.
- Đề nói "every occurrence" ⇒ đếm theo **vị trí bắt đầu**, không theo nội dung.

## 14. Biến thể

1. **Độ dài `k` bất kỳ** (LeetCode 1100 — Find K-Length Substrings With No Repeated Characters).
2. **Chuỗi con dài nhất không lặp** (LeetCode 3) ⇒ cửa sổ trượt **co giãn**.
3. **Mảng con độ dài `k` có tổng lớn nhất** (LeetCode 643) ⇒ cửa sổ cố định với tổng.
4. **Đếm chuỗi con độ dài `k` có đúng `m` ký tự khác nhau**.

In [27]:
def max_avg_subarray(nums, k):
    window = sum(nums[:k])
    best = window
    for i in range(k, len(nums)):
        window += nums[i] - nums[i - k]      # thêm phải, bỏ trái
        best = max(best, window)
    return best / k

print("LC643 [1,12,-5,-6,50,3], k=4 ->", max_avg_subarray([1, 12, -5, -6, 50, 3], 4))
print("LC1100 'havefunonleetcode', k=5 ->", count_good_window("havefunonleetcode", 5))

LC643 [1,12,-5,-6,50,3], k=4 -> 12.75
LC1100 'havefunonleetcode', k=5 -> 6


## 15. Bài tập (tự làm trước)

**Bài 1 — tính tay:** liệt kê mọi chuỗi con độ dài 3 của `"owuxoelszb"` có ký tự lặp. Đáp án của bài là bao nhiêu?

**Bài 2 — dự đoán output:**

```python
s = "abca"
print(list(range(len(s) - 2)), [s[i:i+3] for i in range(len(s) - 2)], len({*"aab"}))
```

**Bài 3 — điền TODO:** so sánh trực tiếp.

In [28]:
def exercise_count_good(s):
    # TODO 1: count = 0
    # TODO 2: for i in range(len(s) - 2)
    # TODO 3: a, b, c = ...; nếu đôi một khác nhau thì count += 1
    pass

# assert exercise_count_good("xyzzaz") == 1
# assert exercise_count_good("aababcabc") == 4

**Bài 4 — tổng quát `k` (LC 1100):** đếm chuỗi con độ dài `k` không lặp ký tự, dùng cửa sổ trượt.

In [29]:
def exercise_count_k_distinct(s, k):
    # TODO: dict đếm, thêm phải, bỏ trái (xóa key về 0), kiểm len(cnt) == k
    pass

# assert exercise_count_k_distinct("havefunonleetcode", 5) == 6
# assert exercise_count_k_distinct("home", 5) == 0

**Bài 5 — LeetCode 3:** độ dài chuỗi con **dài nhất** không lặp ký tự.

In [30]:
def exercise_longest_unique(s):
    # TODO: cửa sổ co giãn; khi s[right] đã có trong cửa sổ thì dịch left
    pass

# assert exercise_longest_unique("abcabcbb") == 3
# assert exercise_longest_unique("bbbbb") == 1
# assert exercise_longest_unique("pwwkew") == 3
# assert exercise_longest_unique("") == 0

**Bài 6 — mini challenge:** đếm số chuỗi con độ dài `k` có **đúng `m`** ký tự khác nhau.

In [31]:
def exercise_count_exact_distinct(s, k, m):
    # TODO: như bài 4 nhưng so len(cnt) == m
    pass

# assert exercise_count_exact_distinct("aababcabc", 3, 2) == 3     # aab, aba, bab
# assert exercise_count_exact_distinct("aaaa", 2, 1) == 3

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | có `10 - 2 = 8` chuỗi con | tìm cặp ký tự trùng trong mỗi bộ 3 | chữ `o` ở vị trí 0 và 4 — có cùng nằm trong một bộ 3 không? |
| 2 | `range(2)` | slicing `[0:3]`, `[1:4]` | `{*"aab"}` = `{'a','b'}` |
| 3 | 3 cặp so sánh | `a != b and b != c and a != c` | `count += 1` |
| 4 | `right >= k` thì bỏ trái | `del cnt[old]` khi về 0 | `right >= k - 1 and len(cnt) == k` |
| 5 | `last[ch]` = vị trí gần nhất | `left = max(left, last[ch] + 1)` | `best = max(best, right - left + 1)` |
| 6 | dùng lại bài 4 | đổi điều kiện | `len(cnt) == m` |

## 17. Đáp án tham khảo

**Bài 1:** các bộ 3 của `"owuxoelszb"`: owu, wux, uxo, xoe, oel, els, lsz, szb — **không** bộ nào có ký tự lặp (chữ `o` cách nhau 4 vị trí) ⇒ đáp án **8**.

**Bài 2:** `[0, 1] ['abc', 'bca'] 2`.

In [32]:
s = "owuxoelszb"
print([s[i:i + 3] for i in range(len(s) - 2)], "->", Solution().countGoodSubstrings(s))
s = "abca"
print(list(range(len(s) - 2)), [s[i:i+3] for i in range(len(s) - 2)], len({*"aab"}))

['owu', 'wux', 'uxo', 'xoe', 'oel', 'els', 'lsz', 'szb'] -> 8
[0, 1] ['abc', 'bca'] 2


`{*"aab"}` "trải" chuỗi vào một set literal — tương đương `set("aab")`.

In [33]:
def answer_count_good(s):
    count = 0
    for i in range(len(s) - 2):
        a, b, c = s[i], s[i + 1], s[i + 2]
        if a != b and b != c and a != c:
            count += 1
    return count


def answer_count_k_distinct(s, k):
    cnt = {}
    count = 0
    for right, ch in enumerate(s):
        cnt[ch] = cnt.get(ch, 0) + 1
        if right >= k:
            old = s[right - k]
            cnt[old] -= 1
            if cnt[old] == 0:
                del cnt[old]
        if right >= k - 1 and len(cnt) == k:
            count += 1
    return count


def answer_longest_unique(s):
    last = {}
    left = best = 0
    for right, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            left = last[ch] + 1
        last[ch] = right
        best = max(best, right - left + 1)
    return best


def answer_count_exact_distinct(s, k, m):
    cnt = {}
    count = 0
    for right, ch in enumerate(s):
        cnt[ch] = cnt.get(ch, 0) + 1
        if right >= k:
            old = s[right - k]
            cnt[old] -= 1
            if cnt[old] == 0:
                del cnt[old]
        if right >= k - 1 and len(cnt) == m:
            count += 1
    return count


assert answer_count_good("xyzzaz") == 1
assert answer_count_good("aababcabc") == 4
assert answer_count_k_distinct("havefunonleetcode", 5) == 6
assert answer_count_k_distinct("home", 5) == 0
assert answer_longest_unique("abcabcbb") == 3
assert answer_longest_unique("bbbbb") == 1
assert answer_longest_unique("pwwkew") == 3
assert answer_longest_unique("") == 0
assert answer_count_exact_distinct("aababcabc", 3, 2) == 3
assert answer_count_exact_distinct("aaaa", 2, 1) == 3
for _ in range(300):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(0, 25)))
    brute = max([j - i for i in range(len(s) + 1) for j in range(i, len(s) + 1) if len(set(s[i:j])) == j - i], default=0)
    assert answer_longest_unique(s) == brute
    k = rng.randint(1, 4)
    assert answer_count_k_distinct(s, k) == sum(len(set(s[i:i + k])) == k for i in range(len(s) - k + 1))
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countGoodSubstrings(self, s: str) -> int:
        count = 0
        for i in range(len(s) - 2):              # mọi vị trí bắt đầu của chuỗi con dài 3
            a, b, c = s[i], s[i + 1], s[i + 2]
            if a != b and b != c and a != c:     # đủ 3 cặp
                count += 1
        return count
```

Khung cửa sổ trượt cố định `k`: **thêm `s[right]` → nếu `right >= k` bỏ `s[right-k]` → nếu `right >= k-1` thì xét cửa sổ.**

Checklist:

- [ ] `range(len(s) - 2)` — không phải `range(len(s))`.
- [ ] So đủ **3 cặp** (hoặc `len(set(...)) == 3`).
- [ ] Đếm **mọi lần xuất hiện**, không dùng set chuỗi con.
- [ ] Chuỗi ngắn hơn 3 ⇒ trả 0 (vòng lặp tự rỗng).
- [ ] Cửa sổ trượt: xóa key khi đếm về 0.